
## Folder 04 — File 02: Silver Products


**Transformation rules:**

- Trim spaces from product names and categories.
- Standardize stock_status to uppercase.
- Convert unit_price and unit_cost to DECIMAL(12,2).
- Reject missing product IDs, missing names, invalid prices, and negative prices or costs.
- Keep one record per product_id.

In [0]:

## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_products"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_products"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_products"
)


## 3. Create Silver schema


spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


## 4. Read Bronze products


if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(
        f"Bronze table not found: {BRONZE_TABLE}. "
        "Run Folder 03 first."
    )

bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze product records: {bronze_df.count()}")

display(bronze_df.limit(10))


## 5. Clean and standardize product columns


cleaned_df = (
    bronze_df

    .withColumn(
        "product_id",
        F.trim(F.col("product_id").cast("string"))
    )
    .withColumn(
        "product_name",
        F.trim(F.col("product_name").cast("string"))
    )
    .withColumn(
        "category",
        F.initcap(F.trim(F.col("category").cast("string")))
    )
    .withColumn(
        "stock_status",
        F.upper(F.trim(F.col("stock_status").cast("string")))
    )

    # Convert prices into decimal values

    .withColumn(
        "unit_price",
        F.expr("try_cast(unit_price AS DECIMAL(12,2))")
    )
    .withColumn(
        "unit_cost",
        F.expr("try_cast(unit_cost AS DECIMAL(12,2))")
    )

    # Convert created_at into DATE

    .withColumn(
        "created_at",
        F.to_date(F.col("created_at").cast("string"))
    )
)

print("Product standardization completed.")

display(cleaned_df.limit(10))


## 6. Identify invalid product records


invalid_condition = (
    F.col("product_id").isNull()
    | (F.col("product_id") == "")
    | F.col("product_name").isNull()
    | (F.col("product_name") == "")
    | F.col("unit_price").isNull()
    | F.col("unit_cost").isNull()
    | (F.col("unit_price") < 0)
    | (F.col("unit_cost") < 0)
)

invalid_df = (
    cleaned_df
    .filter(invalid_condition)
    .withColumn(
        "_quarantine_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("product_id").isNull()
                | (F.col("product_id") == ""),
                F.lit("Missing or empty product_id")
            ),
            F.when(
                F.col("product_name").isNull()
                | (F.col("product_name") == ""),
                F.lit("Missing product_name")
            ),
            F.when(
                F.col("unit_price").isNull(),
                F.lit("Missing or invalid unit_price")
            ),
            F.when(
                F.col("unit_cost").isNull(),
                F.lit("Missing or invalid unit_cost")
            ),
            F.when(
                F.col("unit_price") < 0,
                F.lit("Negative unit_price")
            ),
            F.when(
                F.col("unit_cost") < 0,
                F.lit("Negative unit_cost")
            )
        )
    )
)

valid_df = cleaned_df.filter(~invalid_condition)

print(f"Valid product records: {valid_df.count()}")
print(f"Invalid product records: {invalid_df.count()}")


## 7. Remove duplicate product records


deduplication_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file").desc_nulls_last()
    )
)

deduplicated_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(deduplication_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    f"Records after deduplication: "
    f"{deduplicated_df.count()}"
)


## 8. Save invalid records to quarantine

(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table created: {QUARANTINE_TABLE}")


## 9. Save clean records to Silver Delta table

(
    deduplicated_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table created: {SILVER_TABLE}")


## 10. Validate Silver product records

silver_df = spark.table(SILVER_TABLE)

total_silver_records = silver_df.count()

duplicate_product_groups = (
    silver_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_price_records = (
    silver_df
    .filter(
        F.col("unit_price").isNull()
        | F.col("unit_cost").isNull()
        | (F.col("unit_price") < 0)
        | (F.col("unit_cost") < 0)
    )
    .count()
)

missing_product_ids = (
    silver_df
    .filter(
        F.col("product_id").isNull()
        | (F.col("product_id") == "")
    )
    .count()
)

print("=" * 50)
print("SILVER PRODUCT VALIDATION")
print("=" * 50)

print(f"Silver records: {total_silver_records}")
print(f"Duplicate product groups: {duplicate_product_groups}")
print(f"Invalid price records: {invalid_price_records}")
print(f"Missing product IDs: {missing_product_ids}")

if (
    duplicate_product_groups > 0
    or invalid_price_records > 0
    or missing_product_ids > 0
):
    raise Exception("Silver product validation failed.")

print("Silver product validation passed.")


## 11. Display final Silver product data

display(
    silver_df.orderBy("product_id").limit(20)
)


## 12. Final summary

print("=" * 50)
print("PRODUCT SILVER TRANSFORMATION COMPLETED")
print("=" * 50)

print(f"Bronze table     : {BRONZE_TABLE}")
print(f"Silver table     : {SILVER_TABLE}")
print(f"Quarantine table : {QUARANTINE_TABLE}")

print(f"Bronze records   : {bronze_df.count()}")
print(f"Silver records   : {total_silver_records}")
print(f"Invalid records  : {invalid_df.count()}")

print("=" * 50)
